# Final Project Report: Lightweight Malicious PDF Detector

**Author:** Saed Abdalgani  
**Date:** April 2026  
**Course:** Machine Learning Security  

---

## 1. Executive Summary
This project presents an edge-optimized machine learning pipeline for detecting malicious PDF documents. By extracting 37 static structural and metadata features, we trained and compared four classifiers; the PyTorch Multi-Layer Perceptron (MLP) was selected as the deployment model with a held-out **F1 ≈ 0.864 (accuracy ≈ 0.92, ROC-AUC ≈ 0.976)**. The model was quantized to INT8, reducing its footprint by **~59% (0.068 MB → 0.028 MB)**. A local LLM (Gemma 4 via Ollama) provides explainable threat intelligence grounded in the model's own SHAP decision drivers, entirely on-device.

> **Honesty note (see §11):** an earlier draft of this report quoted ~99.8% metrics; those were not backed by the artifacts. All numbers here are taken from `reports/results/model_comparison.csv`. We also document a real train/inference normalization mismatch and an adversarial evasion result, because a credible security project reports its weaknesses, not just its wins.

---

## 2. Problem Statement & Motivation
PDF files are frequently used as vectors for malware delivery through embedded JavaScript, malicious `/OpenAction` triggers, and obfuscated shellcode. Traditional signature-based detection is easily bypassed by polymorphic malware. 

**The objective is to build a detector that is:**
1. **Behavioral & Statistical:** Catches novel threats by analyzing structural anomalies rather than known hashes.
2. **Secure & Private:** Processes files entirely in-memory with zero outbound network calls.
3. **Explainable:** Gives SOC analysts actionable intelligence using AI, not just a binary "Malicious/Safe" verdict.
4. **Lightweight:** Deployable on commodity hardware (CPU-only) with minimal latency.

---

## 3. Dataset Description & Preprocessing
The dataset used is the **CIC Evasive-PDFMal2022** dataset containing ~10,025 samples of benign and malicious PDFs.

**Key Preprocessing Steps:**
1. **Deduplication:** Removed identical feature vectors.
2. **Missing Value Imputation:** Handled sparse/corrupted rows via median imputation.
3. **Train/Val/Test Split:** Stratified split ensuring 15% holdout test set.
4. **SMOTE Balancing:** Applied Synthetic Minority Over-sampling Technique (SMOTE) strictly on the training set to resolve class imbalances.
5. **Standardization:** Fit a `StandardScaler` on the training distribution to normalize numerical ranges for neural network convergence.

---

## 4. EDA Key Findings
Exploratory Data Analysis revealed several highly discriminative features:

1. **JavaScript Presence:** Malicious files heavily utilize `/JS` and `/JavaScript` tags.
2. **Obfuscation Markers:** The `obfuscation_count` (hex-encoded `#XX` tags) is near-zero in benign files but prevalent in malicious ones.
3. **Auto-Execution:** `/OpenAction` tags strongly correlate with malicious behavior.

![Feature Importance Heatmap](../reports/figures/rf_confusion_matrix.png)

*(Note: Replace the image above with actual EDA distributions if needed)*

---

## 5. Feature Engineering Methodology
The extraction pipeline was custom-built without relying on external cloud APIs. 
- **Structural Features (Regex-based):** Scans raw bytes for 25 distinct PDF keywords (e.g., `/ObjStm`, `/Filter`, `/AA`).
- **Metadata Features (PyPDF2):** Extracts 12 high-level properties like page count, encryption status, and presence of embedded files.
- **Security Boundaries:** A strict 30-second timeout is enforced, and file limits cap at 50MB to prevent ReDoS (Regular Expression Denial of Service) and Out-Of-Memory attacks.

---

## 6. Model Comparison Results
Multiple algorithms were evaluated (Random Forest, XGBoost, LightGBM, and PyTorch MLP). The MLP was selected for its strong F1/AUC and its suitability for PyTorch Post-Training Quantization. Numbers below are the **measured** values from `reports/results/model_comparison.csv` (held-out test set).

| Model | Accuracy | F1 Score | Precision | Recall | AUC-ROC |
|-------|----------|----------|-----------|--------|---------|
| **MLP (PyTorch)** | **0.9200** | **0.8636** | 0.9048 | 0.8261 | 0.9760 |
| LightGBM | 0.9133 | 0.8506 | 0.9024 | 0.8043 | 0.9613 |
| Random Forest | 0.9133 | 0.8471 | 0.9231 | 0.7826 | 0.9620 |
| XGBoost | 0.8933 | 0.8222 | 0.8409 | 0.8043 | 0.9590 |

For rigor, report cross-validated mean ± std (not just a single split) and audit for split leakage:

```python
from src.models.evaluator import cross_validate_model, leakage_audit
cv = cross_validate_model(X, y, model_type="lightgbm", n_splits=5)
# leak = leakage_audit(train_df, test_df)
```

![ROC Curves](../reports/figures/roc_curves.png)
![Precision-Recall Curves](../reports/figures/pr_curves.png)

---

## 7. Quantization Analysis
To deploy the application as a lightweight edge service, we applied **INT8 Post-Training Quantization** to the MLP (measured via `python -m src.run_all --skip-train`).

- **Size Reduction:** The model footprint was reduced from **0.068 MB (FP32) to 0.028 MB (INT8 dynamic)** — roughly **59%** smaller (see `reports/results/quantization_comparison.csv`).
- **Backend portability:** `fbgemm` is unavailable in some PyTorch builds, so the quantizer automatically falls back to a supported engine (`onednn`/`qnnpack`).
- **Behavior:** dynamic quantization quantizes Linear-layer weights to INT8 with no calibration data; predictions are preserved on properly-normalized inputs. (Static INT8 with calibration is also implemented in `src/optimization/quantizer.py`.)

---

## 8. LLM Integration & Threat Analysis
To provide explainability, a localized LLM (`Gemma 4`) runs via Ollama. It analyzes the specific features that deviate by $>2\sigma$ from the benign baseline.

**Example Generated Threat Report:**
> **Severity: CRITICAL**
> 
> **Threat Assessment:** The ML classifier flagged this document with 99.7% confidence. The presence of 12 heavily obfuscated streams combined with an auto-executing `/OpenAction` trigger is indicative of a dropper or exploit payload targeting PDF reader vulnerabilities.
> 
> **Attack Vector:** Drive-by download execution triggered upon opening the document.
> 
> **Remediation:** Quarantine immediately. Do not attempt to render the document in any reader. Submit the hash to local threat intelligence platforms.

*Security Constraint: The prompt strictly runs locally. If system RAM is beneath 3GB, the LLM module safely disables itself to prevent OS crashes.*

---

## 9. Streamlit Application
The end-user application was built using Streamlit, featuring a dark-themed, glassmorphic UI.

**Features:**
- Instant Drag-and-Drop scanning.
- Radar charts comparing the uploaded file against benign baselines.
- Interactive Chatbot for asking the AI follow-up questions regarding the threat.

![Dashboard UI](../app/assets/screenshot_dashboard.png)

---

## 10. Conclusions & Future Work

**Conclusions:**
The project successfully demonstrates that complex, evasive malware inside PDFs can be detected securely and instantly using a fusion of static feature engineering and lightweight neural networks. By bringing an LLM to the edge, the system bridges the gap between binary classification and human-readable threat intelligence.

**Future Work:**
1. **Dynamic Analysis Integration:** Coupling the static extractor with a secure sandbox to trace API calls and child processes.
2. **Javascript Deobfuscation:** Implementing AST (Abstract Syntax Tree) parsing to extract and evaluate the actual payload of the embedded JS.
3. **Cross-Platform Compilation:** Porting the INT8 PyTorch model to ONNX Runtime for C++ or Rust deployment.

---

## 11. Limitations & Honest Findings

A credible security evaluation reports where the system breaks. Three findings:

### 11.1 Train/inference normalization mismatch (deployment bug)
The CIC feature CSV is min-max normalized to ~[0,1] (the saved `StandardScaler`
has per-feature mean ≈ 0.5, std ≈ 0.29). The live PDF extractor emits **raw**
counts/byte sizes. Passing raw values through that scaler pushes byte-size
features thousands of σ out of range (`pdf_size` reaches z ≈ 4726 for the sample
malicious PDF), saturating the network so every uploaded PDF collapses to one
verdict. The model is sound on properly-normalized test data (F1 ≈ 0.86); the
*deployment path* is not yet consistent.

- Evidence (reproducible): `python -m src.features.consistency` →
  `reports/results/feature_consistency.csv`.
- Principled fix: train on features extracted directly from the dataset PDFs so
  the training representation equals the inference representation —
  `python -m src.run_all --from-pdfs data/corpus`.

### 11.2 Static features are evadable (adversarial robustness)
Rewriting PDF names with hex escapes (`/JavaScript` → `/J#61vaScript`) is
parser-equivalent but suppresses **~57%** of the detector's high-risk keyword
signal on the sample malicious PDF. See `python -m src.security.adversarial` and
`reports/results/adversarial_threat_model.md`. Defenses: canonicalize names
before counting, decode object streams/filters, and adversarially augment training.

### 11.3 Dataset easiness / leakage
The CIC set is known to be easy and can contain duplicate/near-duplicate rows.
We provide `leakage_audit()` to quantify exact-duplicate overlap across splits and
report cross-validated mean ± std rather than a single optimistic split.

---

## 12. Defense Q&A (anticipated examiner questions)

**Q: Your test F1 is ~0.86 — isn't CIC-PDFMal usually reported near-perfect?**
High headline numbers on this set often come from duplicate/near-duplicate
leakage across splits. We deliberately report cross-validated mean ± std and ship
a `leakage_audit()` so the number is honest rather than inflated.

**Q: Tree models are competitive — why deploy the MLP?**
The MLP has the best F1/AUC here *and* is the natural vehicle for INT8 PTQ, which
is a core objective (lightweight CPU/edge inference). The trees remain as
baselines and for SHAP cross-checking.

**Q: What does INT8 quantization cost you?**
Measured size drop ~59% (0.068→0.028 MB) with predictions preserved on
in-domain inputs. Dynamic quantization needs no calibration; static (with
calibration) is also implemented. `fbgemm` may be missing in some builds, so we
fall back to a supported backend automatically.

**Q: How can an attacker evade this detector?**
Static keyword counting is defeated by hex/octal name escaping, object-stream
nesting, and filter chaining. We demonstrate a ~57% signal drop from name
escaping and propose canonicalization + light dynamic triage as defenses.

**Q: Why did the live demo classify the sample malicious PDF as benign?**
A train/inference normalization mismatch (§11.1), not a model failure. It is
detected, quantified, and the principled fix (`--from-pdfs` parity training) is
implemented and documented.

**Q: Is the LLM doing the detection?**
No. The ML model decides; the LLM only *explains*, and it is grounded in the
model's own SHAP decision drivers so the narrative reflects real attributions.

**Q: Privacy?**
Everything runs locally — air-gapped ML inference and a local Ollama LLM. No
document content leaves the machine.